<a href="https://colab.research.google.com/github/DrumScript/DrumScript/blob/main/docs/guide/interactive/drumscript_interactive_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🥁 DrumScript - Combined

This notebook combines several functions of DrumScript into one notebook

**DrumScript** is an open-source Python library and CLI tool for drum audio analysis and transcription.

Give it a recording — a full mix or an isolated drum stem — and it will generate PDF sheet music, MIDI files, and MusicXML output.

This notebook walks through the four core capabilities:

| # | Feature | What it does |
|---|---------|-------------|
| 1 | **Tempo Detection** | `DrumScript` stimates BPM from drum audio file based on tempogram-anaysis |
| 2 | **Stem Separation** | Extracts drums from `.wav` and `.mp3` songs. Also supports extraction of bass, vocals, and other instruments |
| 3 | **Backing Tracks** | Give `DrumScript` a song and it will mute the drums to create a backing track for you|
| 4 | **Transcription** | `DrumScript` converts drum audio → PDF sheet music . The --full flag also means you can give it a full song and it will extract the drums **and** transcribe|

> **Alpha Notice:** DrumScript is currently in **public alpha** (v0.1.4). The API is stabilising but may change before v1.0. If you encounter issues, please [open an issue on GitHub](https://github.com/DrumScript/DrumScript/issues).

<!--date_added:saturday-17-jan-2026-->
<!--date_updated:fri-05-june-2026-->

---

## 0. Setup

Install DrumScript and system dependencies, then download the demo audio.

In [1]:
import drumscript as ds

In [2]:
# ── Run this cell after restarting ───────────────────────────────
import os

import IPython.display as ipd

print(f"DrumScript v{ds.__version__} ready.")

DrumScript v0.1.5 ready.


---
## Load Audio

You can either load demo audio or upload your own

1. Load demo audio

In [3]:
# ── Download demo audio from the DrumScript repo ─────────────────
AUDIO_FILE = "./audio/test_track_1.wav"

ds.load_audio(AUDIO_FILE)
# Listen to the input audio
print("Note: Credit to https://pixabay.com/music/search/warbringer/ for the royalty-free music")
print("\n▶ Demo audio:")
ipd.display(ipd.Audio(AUDIO_FILE))

Note: Credit to https://pixabay.com/music/search/warbringer/ for the royalty-free music

▶ Demo audio:


2. Load your own audio


In [4]:
# ── Audio Upload Widget ──────────────────────────────────────────
import ipywidgets as widgets
from IPython.display import Audio, clear_output, display

# File-type selector — controls which demo file your upload replaces
audio_type = widgets.RadioButtons(
    options=[
        ("Drum-only stem (replaces DRUM_ONLY_FILE → used in section 2a)", "drum_only"),
        ("Full polyphonic song (replaces AUDIO_FILE → used in section 2b, stems, backing track)", "full_song"),
    ],
    value="drum_only",
    description="Upload type:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="auto"),
)

upload_btn = widgets.FileUpload(
    accept=".wav,.mp3,.flac,.ogg",
    multiple=False,
    description="Upload Audio",
    button_style="info",
    layout=widgets.Layout(width="300px"),
)

output = widgets.Output()


def on_upload(change):
    """Save the uploaded file and override the appropriate demo-file variable."""
    global DRUM_ONLY_FILE, AUDIO_FILE, FULL_SONG_FILE
    with output:
        clear_output()
        for filename, file_info in upload_btn.value.items():
            uploaded_path = f"/content/{filename}"
            with open(uploaded_path, "wb") as f:
                f.write(file_info["content"])
            print(f"✅ Uploaded: {filename}")
            print(f"📁 Saved to: {uploaded_path}")

            # Override the appropriate global variable so downstream cells pick it up
            if audio_type.value == "drum_only":
                DRUM_ONLY_FILE = uploaded_path
                print("\n→ This file is now DRUM_ONLY_FILE.")
                print("  Re-run transcription section 2a to use it.")
            else:
                AUDIO_FILE = uploaded_path
                FULL_SONG_FILE = uploaded_path
                print("\n→ This file is now AUDIO_FILE (and FULL_SONG_FILE).")
                print("  Re-run transcription section 2b, stem separation, or the backing track to use it.")

            display(Audio(uploaded_path))


upload_btn.observe(on_upload, names="value")

print("Optional: upload your own drum recording or song.")
print("If you skip this, the demo audio downloaded above will be used.\n")
display(audio_type, upload_btn, output)

Optional: upload your own drum recording or song.
If you skip this, the demo audio downloaded above will be used.



RadioButtons(description='Upload type:', layout=Layout(width='auto'), options=(('Drum-only stem (replaces DRUM…

FileUpload(value=(), accept='.wav,.mp3,.flac,.ogg', button_style='info', description='Upload Audio', layout=La…

Output()

---

## 1. Tempo Detection

DrumScript estimates BPM using a **spectral onset envelope** approach:

1. Computes an onset strength envelope (peaks wherever a drum hit occurs)
2. Builds a tempogram measuring periodicity at every candidate BPM
3. Picks the BPM with the strongest peak within a musically plausible range (60–240 BPM)


One line of code:

In [5]:
bpm = ds.detect_tempo(AUDIO_FILE)
print(f"Detected tempo: {bpm:.1f} BPM")

Detected tempo: 99.4 BPM


You can also get detailed stats by passing `full=True`:

In [6]:
stats = ds.detect_tempo(AUDIO_FILE, full=True)
# print(f"BPM: {stats['bpm']:.1f}")
print(f"Sample rate: {stats['sr']} Hz")

Sample rate: 44100 Hz


---

## 2. Transcription — Audio to Sheet Music

The full transcription pipeline:

**Audio → Load & Normalise → Detect Tempo → Detect Onsets → Extract Features → Classify Hits → Build Score → PDF**

The classifier is **deterministic and rule-based** — it uses physics-derived frequency bands, spectral centroid, decay time, and energy ratios to identify kick, snare, hi-hat (open/closed), toms, crash, and ride.

No ML model required.

In [7]:
# Run end-to-end transcription with full output
result = ds.transcribe(AUDIO_FILE, output_dir="outputs", full_song=True, full=True)

print(f"\nPDF saved to:  {result['pdf_path']}")
print(f"Detected tempo: {result['tempo']:.1f} BPM")
print(f"Total onsets:   {len(result['onsets'])}")
print(f"Classified events: {len(result['events'])}")

...Separating drum stem...
Starting Demucs separation for: ./audio/test_track_1.wav...
Demucs separation finished in 1.44 minutes.
Drum stem extracted successfully to: /Users/victoriamckinney/Library/Visual Studio Projects/DrumScript/docs/guide/interactive/htdemucs/test_track_1/drums.wav
...Loading & analysing audio...
(HOP_LENGTH: 128)
(Wait Frames Applied: 17)
(len_onset_times:1123)
   -> Tempo: 99.4 BPM | Onsets: 1123
   -> Classifier: standard polyphonic engine
   -> Classified 1123 events
...Building score: outputs/test_track_1_transcription.pdf
--- Building Score for: outputs/test_track_1_transcription.pdf [Time Sig: 4/4] ---
Saving to: outputs/test_track_1_transcription.json
Generating PDF: outputs/test_track_1_transcription.pdf (Sig: 4/4, 99 BPM)
PDF successfully saved to: outputs/test_track_1_transcription.pdf
SUCCESS: MIDI file generated at -> outputs/test_track_1_transcription.mid
--- Done ---

PDF saved to:  outputs/test_track_1_transcription.pdf
Detected tempo: 99.4 BPM
To

/content/outputs/test_track_1_transcription.pdf

### Inspect the classification results

Each event in `result['events']` contains the timestamp, classified instruments, and the raw DSP features used by the classifier.

In [8]:
# Show the first 10 classified events
for i, event in enumerate(result["events"][:10]):
    time = event["time_sec"]
    instruments = ", ".join(event["instruments"])
    print(f"  {time:6.3f}s  →  {instruments}")

   5.132s  →  kick, hi_hat_open
  10.565s  →  snare, hi_hat_closed
  10.716s  →  hi_hat_closed
  10.716s  →  snare
  10.867s  →  snare
  11.018s  →  kick
  11.169s  →  kick
  11.169s  →  unknown
  11.320s  →  unknown
  11.471s  →  kick


---

## 3. Extract drums from any track! (`.wav` and `.mp3`^)

**Stem Separation**

DrumScript uses [Demucs](https://github.com/adefossez/demucs) (Meta's `htdemucs` 4-stem model) to separate any audio into:

- **Drums**
- **Bass**
- **Vocals**
- **Other** (guitars, keys, etc.)

**Note:** Stem separation is *somewhat* computationally intensive. On a free Colab CPU, a 3-minute track may take several minutes. Use a GPU runtime (Runtime → Change runtime type → T4 GPU) for faster results.

  > ###### ^ `.mp3` requires ffmpeg installation (`brew install ffmpeg`)*

In [ ]:
# Extract all stems
stem_result = ds.extract_stems(
    AUDIO_FILE,
    all_stems=True,
    output_dir="stems",
    full=True,
)

print("Stem separation complete.")
print(f"Output directory: {stem_result['output_directory']}")

Starting Demucs separation for: test_track_1.wav...


In [ ]:
# List the extracted stems
stems_dir = stem_result["output_directory"]
stem_files = sorted([f for f in os.listdir(stems_dir) if f.endswith(".wav")])

for f in stem_files:
    path = os.path.join(stems_dir, f)
    print(f"\n▶ {f}")
    ipd.display(ipd.Audio(path))

## **Listen to the stems**

> **Original audio**


In [ ]:
print("Note: We are using a copyright-free synthetic track for this demonstration.")
print("\n▶ Demo audio:")
ipd.display(ipd.Audio(AUDIO_FILE))

> ### Drums

In [ ]:
# print(f"Note: We are using a copyright-free synthetic track for this demonstration.")
print("\nDrums:")
# ipd.display(ipd.Audio("stems/test_track_1/test_track_1_only_drums.wav"))
print("\n")
display(Audio("stems/test_track_1/test_track_1_only_drums.wav"))

> ### Bass guitar

In [ ]:
# print(f"Note: We are using a copyright-free synthetic track for this demonstration.")
print("\nBass guitar:")
print("\n")
display(Audio("stems/test_track_1/test_track_1_only_bass.wav"))


> ### Vocals


In [ ]:
# print(f"Note: We are using a copyright-free synthetic track for this demonstration.")
print("\nVocals:")
print("\n")
display(Audio("stems/test_track_1/test_track_1_only_vocals.wav"))

---

## 4. Give `DrumScript` any song (in `.mp3` or `.wav`) and make a backing track (it even extracts it for you first if it's a multi-instrumental song)

**Backing track for drummers**

In [ ]:
# Create a drumless backing track
backing_result = ds.extract_stems(
    AUDIO_FILE,
    drumless=True,
    output_dir="backing_track",
    full=True,
)

print("Backing track created.")
print(f"Output directory: {backing_result['output_directory']}")

In [ ]:
# 3. Playback!
# Workaround: extract_stems currently returns the drum path
# even with drumless=True. Manually find the backing track.
import os

backing_dir = os.path.join("backing_track", "test_track_1")
backing_files = [f for f in os.listdir(backing_dir) if "no_drums" in f]

if backing_files:
    backing_path = os.path.join(backing_dir, backing_files[0])
    print("Created backing track:", backing_path)
    display(Audio(backing_path))
else:
    print("No backing track found — check the output directory.")

---
## What's next for `DrumScript`?

*DrumScript is in public alpha (June–August 2026). Feedback and contributions welcome.*


---

## Other useful things

**CLI usage** (works in any terminal):

```bash
pip install drumscript
drumscript your_audio.wav              # transcribe a drum stem
drumscript full_song.mp3 --full        # full song (auto-separates drums)
drumscript full_song.mp3 --drumless    # create a backing track
```

**Links:**

- 📖 [Documentation](https://drumscript.github.io/DrumScript/)
- 💻 [GitHub](https://github.com/DrumScript/DrumScript)
- 📦 [PyPI](https://pypi.org/project/drumscript/)
- 🐛 [Report an issue](https://github.com/DrumScript/DrumScript/issues)
- 📖 [More Runbooks](https://drumscript.github.io/DrumScript/guide/interactive/index.html)

>

**&copy;  DrumScript, Apache License, 2026**
>
**hello.drumscript@gmail.com**

---

<!--END-->